In [ ]:
from __future__ import annotations

import math
import os
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from tqdm import tqdm
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
BLENDSHAPES = [
    "_neutral", "browDownLeft", "browDownRight", "browInnerUp",
    "browOuterUpLeft", "browOuterUpRight", "cheekPuff",
    "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft",
    "eyeBlinkRight", "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft",
    "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
    "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
    "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose",
    "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft",
    "mouthFrownRight", "mouthFunnel", "mouthLeft",
    "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
    "mouthPressRight", "mouthPucker", "mouthRight",
    "mouthRollLower", "mouthRollUpper", "mouthShrugLower",
    "mouthShrugUpper", "mouthSmileLeft", "mouthSmileRight",
    "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft",
    "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight",
]

POSE_HAND_LANDMARKS = [
    "left_hand_WRIST", "right_hand_WRIST",
    "pose_LEFT_ELBOW", "pose_RIGHT_ELBOW",
    "pose_LEFT_HIP", "pose_RIGHT_HIP",
    "pose_LEFT_SHOULDER", "pose_RIGHT_SHOULDER",
    "pose_NOSE",
]

ANGLES = [
    "head_pitch_deg", "head_yaw_deg", "head_roll_deg",
    "left_arm_angle", "right_arm_angle",
    "torso_pitch", "torso_roll", "torso_yaw",
]

DISTANCES = [
    "dist_wrist_lr",
    "dist_left_wrist_to_left_shoulder",
    "dist_right_wrist_to_right_shoulder",
    "dist_elbows_lr",
    "dist_left_wrist_to_nose",
    "dist_right_wrist_to_nose",
    "dist_left_wrist_to_right_shoulder",
    "dist_right_wrist_to_left_shoulder",
]

POSE_KEYS = [
    "pose_NOSE",
    "pose_LEFT_SHOULDER", "pose_RIGHT_SHOULDER",
    "pose_LEFT_ELBOW", "pose_RIGHT_ELBOW",
    "pose_LEFT_WRIST", "pose_RIGHT_WRIST",
    "pose_LEFT_HIP", "pose_RIGHT_HIP",
]

HAND_WRISTS = ["left_hand_WRIST", "right_hand_WRIST"]

FACE_MIN_WIDTH  = 50
FACE_MIN_HEIGHT = 50

In [ ]:
base_options  = python.BaseOptions(model_asset_path='model/face_landmarker_v2_with_blendshapes.task')
options       = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=1,
)
face_landmarker = vision.FaceLandmarker.create_from_options(options)
mp_holistic     = mp.solutions.holistic

In [ ]:
def _clamp(value, min_value, max_value):
    return max(min_value, min(value, max_value))


def extract_landmarks(landmarks, prefix, num_landmarks):
    # Extract x/y/z coordinates from a MediaPipe landmark list
    if landmarks:
        return {
            f"{prefix}_{axis}_{i+1}": getattr(landmark, axis)
            for i, landmark in enumerate(landmarks.landmark[:num_landmarks])
            for axis in ['x', 'y', 'z']
        }
    return {
        f"{prefix}_{axis}_{i+1}": np.nan
        for i in range(num_landmarks)
        for axis in ['x', 'y', 'z']
    }


def crop_face_region(frame_in: np.ndarray, body_landmarks: dict) -> np.ndarray:
    # Crop a square face region using shoulder/nose pose landmarks
    r_shoulder_x, r_shoulder_y, _ = body_landmarks.get("right_shoulder", [0, 0, 0])
    l_shoulder_x, l_shoulder_y, _ = body_landmarks.get("left_shoulder",  [0, 0, 0])
    nose_tip_x,   nose_tip_y,   _ = body_landmarks.get("nose",           [0, 0, 0])

    s_x = (r_shoulder_x + l_shoulder_x) / 2
    s_y = (r_shoulder_y + l_shoulder_y) / 2
    sn  = int(math.sqrt((nose_tip_x - s_x) ** 2 + (nose_tip_y - s_y) ** 2) * 0.75)

    h, w, _ = frame_in.shape
    x1 = _clamp(nose_tip_x - sn, 0, w - 1)
    x2 = _clamp(nose_tip_x + sn, 0, w - 1)
    y1 = _clamp(nose_tip_y - sn, 0, h - 1)
    y2 = _clamp(nose_tip_y + sn, 0, h - 1)

    return frame_in[y1:y2, x1:x2, :].copy()


def find_video(root_folder: str) -> list[str]:
    # Recursively find all Front.mp4 files under root_folder
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(root_folder)
        for file in files
        if file == 'Front.mp4'
    ]

In [ ]:
def process_video(video: str):
    holistic_model         = mp_holistic.Holistic(static_image_mode=False, model_complexity=1)
    cap                    = cv2.VideoCapture(video)
    data                   = []
    face_blendshapes_names = None

    frame_count  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    duration_sec = frame_count / fps if fps > 0 else np.nan

    video_metadata = {
        "video_name":   os.path.basename(video),
        "frame_count":  frame_count,
        "fps":          fps,
        "duration_sec": duration_sec,
    }

    for frame_num in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            # Handle dropped frame
            frame_data = {'frame': frame_num + 1}
            frame_data.update({name: np.nan for name in frame_data})
            data.append(frame_data)
            continue

        image      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_data = {'frame': frame_num + 1}

        results = holistic_model.process(image)

        if results.pose_landmarks is None:
            frame_data.update({name: np.nan for name in frame_data})
            data.append(frame_data)
            continue

        # Hand landmarks
        for side, hand_lms in (
            ("left",  results.left_hand_landmarks),
            ("right", results.right_hand_landmarks),
        ):
            if hand_lms:
                for i, landmark in enumerate(hand_lms.landmark):
                    hand_name = mp_holistic.HandLandmark(i).name
                    frame_data[f"{side}_hand_{hand_name}_x"] = landmark.x
                    frame_data[f"{side}_hand_{hand_name}_y"] = landmark.y
                    frame_data[f"{side}_hand_{hand_name}_z"] = landmark.z

        # Pose landmarks (first 25)
        pose_landmarks = results.pose_landmarks
        for i in range(25):
            try:
                landmark  = pose_landmarks.landmark[i]
                pose_name = mp_holistic.PoseLandmark(i).name
                frame_data[f"pose_{pose_name}_x"] = landmark.x
                frame_data[f"pose_{pose_name}_y"] = landmark.y
                frame_data[f"pose_{pose_name}_z"] = landmark.z
            except IndexError:
                frame_data[f"pose_{i}"] = np.nan

        # Convert body keypoints to pixel coords for face cropping
        h, w, _ = frame.shape
        body_landmarks = {
            "right_shoulder": pose_landmarks.landmark[mp_holistic.PoseLandmark.RIGHT_SHOULDER],
            "left_shoulder":  pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_SHOULDER],
            "nose":           pose_landmarks.landmark[mp_holistic.PoseLandmark.NOSE],
        }
        body_landmarks = {
            k: (int(lm.x * w), int(lm.y * h), int(lm.z * w))
            for k, lm in body_landmarks.items()
        }

        face_crop = crop_face_region(frame, body_landmarks)
        face_h, face_w, _ = face_crop.shape

        if face_w >= FACE_MIN_WIDTH and face_h >= FACE_MIN_HEIGHT:
            mp_image     = mp.Image(image_format=mp.ImageFormat.SRGB,
                                    data=cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
            face_results = face_landmarker.detect(mp_image)

            if face_results.face_blendshapes:
                # Transformation matrix
                if face_results.facial_transformation_matrixes:
                    flat = np.array(face_results.facial_transformation_matrixes[0]).reshape(4, 4).flatten()
                    for i, val in enumerate(flat):
                        frame_data[f"transform_{i}"] = val
                else:
                    for i in range(16):
                        frame_data[f"transform_{i}"] = np.nan

                # Blendshape scores
                if face_blendshapes_names is None:
                    face_blendshapes_names = [
                        bs.category_name for bs in face_results.face_blendshapes[0]
                    ]
                scores = [bs.score for bs in face_results.face_blendshapes[0]]
                frame_data.update(dict(zip(face_blendshapes_names, scores)))
            else:
                if face_blendshapes_names:
                    frame_data.update({name: np.nan for name in face_blendshapes_names})

        data.append(frame_data)

    cap.release()
    holistic_model.close()
    return pd.DataFrame(data), video_metadata

In [ ]:
def interpolate_missing_values(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df.interpolate(method='linear', axis=0, limit_direction='both')


def compute_derivatives(df: pd.DataFrame, landmarks: list[str],
                        axes: list[str] = ['x', 'y', 'z']) -> pd.DataFrame:
    # Add velocity and acceleration columns for each landmark/axis pair
    new_cols = {}
    for lm in landmarks:
        for axis in axes:
            col = f"{lm}_{axis}"
            if col in df.columns:
                s = df[col].diff().fillna(0)
                new_cols[f"{col}_velocity"]     = s
                new_cols[f"{col}_acceleration"] = s.diff().fillna(0)
    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df


def compute_stats(df: pd.DataFrame, landmarks: list[str], blendshapes: list[str],
                  axes: list[str] = ['x', 'y', 'z']) -> pd.DataFrame:
    # Return a single-row DataFrame of mean/std for landmarks and blendshapes
    stats = {}
    for lm in landmarks:
        for axis in axes:
            for suffix in (f"{lm}_{axis}", f"{lm}_{axis}_velocity", f"{lm}_{axis}_acceleration"):
                if suffix in df.columns:
                    stats[f"{suffix}_mean"] = df[suffix].mean()
                    stats[f"{suffix}_std"]  = df[suffix].std()
    for bs in blendshapes:
        if bs in df.columns:
            stats[f"{bs}_mean"] = df[bs].mean()
            stats[f"{bs}_std"]  = df[bs].std()
    return pd.DataFrame([stats])

In [ ]:
def compute_transform_features(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    transform_cols     = [f"transform_{i}" for i in range(16)]
    has_face_transform = all(col in df.columns for col in transform_cols)

    def rotation_matrix_to_euler_angles(R):
        sy       = np.sqrt(R[0, 0] ** 2 + R[1, 0] ** 2)
        singular = sy < 1e-6
        if not singular:
            x = np.arctan2(R[2, 1], R[2, 2])
            y = np.arctan2(-R[2, 0], sy)
            z = np.arctan2(R[1, 0], R[0, 0])
        else:
            x = np.arctan2(-R[1, 2], R[1, 1])
            y = np.arctan2(-R[2, 0], sy)
            z = 0
        return np.degrees([x, y, z])

    def angle_between(v1, v2):
        v1, v2 = np.array(v1), np.array(v2)
        norm   = np.linalg.norm(v1) * np.linalg.norm(v2)
        if norm == 0:
            return np.nan
        return np.degrees(np.arccos(np.clip(np.dot(v1, v2) / norm, -1.0, 1.0)))

    head_pitch, head_yaw, head_roll    = [], [], []
    left_arm_angle, right_arm_angle    = [], []
    torso_pitch, torso_roll, torso_yaw = [], [], []

    prev_t = None

    for _, row in df.iterrows():
        # Head rotation from facial transform matrix
        if has_face_transform:
            try:
                mat   = row[transform_cols].to_numpy().reshape(4, 4)
                R     = mat[:3, :3]
                t     = mat[:3, 3]
                pitch, yaw, roll = rotation_matrix_to_euler_angles(R)
                prev_t = t
            except Exception:
                pitch, yaw, roll = np.nan, np.nan, np.nan
        else:
            pitch, yaw, roll = np.nan, np.nan, np.nan

        head_pitch.append(pitch)
        head_yaw.append(yaw)
        head_roll.append(roll)

        # Arm and torso angles from pose landmarks
        try:
            l_shoulder = np.array([row["pose_LEFT_SHOULDER_x"],  row["pose_LEFT_SHOULDER_y"],  row["pose_LEFT_SHOULDER_z"]])
            l_elbow    = np.array([row["pose_LEFT_ELBOW_x"],     row["pose_LEFT_ELBOW_y"],     row["pose_LEFT_ELBOW_z"]])
            r_shoulder = np.array([row["pose_RIGHT_SHOULDER_x"], row["pose_RIGHT_SHOULDER_y"], row["pose_RIGHT_SHOULDER_z"]])
            r_elbow    = np.array([row["pose_RIGHT_ELBOW_x"],    row["pose_RIGHT_ELBOW_y"],    row["pose_RIGHT_ELBOW_z"]])
            l_hip      = np.array([row["pose_LEFT_HIP_x"],       row["pose_LEFT_HIP_y"],       row["pose_LEFT_HIP_z"]])
            r_hip      = np.array([row["pose_RIGHT_HIP_x"],      row["pose_RIGHT_HIP_y"],      row["pose_RIGHT_HIP_z"]])

            torso_vec_vert  = ((l_hip + r_hip) / 2) - ((l_shoulder + r_shoulder) / 2)
            torso_vec_horiz = r_shoulder - l_shoulder

            left_angle        = angle_between(l_elbow - l_shoulder, torso_vec_vert)
            right_angle       = angle_between(r_elbow - r_shoulder, torso_vec_vert)
            torso_pitch_angle = angle_between(torso_vec_vert,  np.array([0, -1, 0]))
            torso_roll_angle  = angle_between(torso_vec_horiz, np.array([1,  0, 0]))
            torso_yaw_angle   = np.degrees(np.arctan2(
                r_shoulder[2] - l_shoulder[2],
                r_shoulder[0] - l_shoulder[0],
            ))
        except Exception:
            left_angle = right_angle = torso_pitch_angle = torso_roll_angle = torso_yaw_angle = np.nan

        left_arm_angle.append(left_angle)
        right_arm_angle.append(right_angle)
        torso_pitch.append(torso_pitch_angle)
        torso_roll.append(torso_roll_angle)
        torso_yaw.append(torso_yaw_angle)

    df["head_pitch_deg"]  = head_pitch
    df["head_yaw_deg"]    = head_yaw
    df["head_roll_deg"]   = head_roll
    df["left_arm_angle"]  = left_arm_angle
    df["right_arm_angle"] = right_arm_angle
    df["torso_pitch"]     = torso_pitch
    df["torso_roll"]      = torso_roll
    df["torso_yaw"]       = torso_yaw

    df.to_csv(csv_path, index=False)
    print(f"Interpolated file with rotations: {csv_path}")
    return df

In [ ]:
def augment_distance(csv_path: str) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"[WARN] Failed to read {csv_path}: {e}")
        return None

    COLS = {
        "L_WRIST":    ("left_hand_WRIST_x",    "left_hand_WRIST_y",    "left_hand_WRIST_z"),
        "R_WRIST":    ("right_hand_WRIST_x",   "right_hand_WRIST_y",   "right_hand_WRIST_z"),
        "L_ELBOW":    ("pose_LEFT_ELBOW_x",    "pose_LEFT_ELBOW_y",    "pose_LEFT_ELBOW_z"),
        "R_ELBOW":    ("pose_RIGHT_ELBOW_x",   "pose_RIGHT_ELBOW_y",   "pose_RIGHT_ELBOW_z"),
        "L_SHOULDER": ("pose_LEFT_SHOULDER_x", "pose_LEFT_SHOULDER_y", "pose_LEFT_SHOULDER_z"),
        "R_SHOULDER": ("pose_RIGHT_SHOULDER_x","pose_RIGHT_SHOULDER_y","pose_RIGHT_SHOULDER_z"),
        "NOSE":       ("pose_NOSE_x",           "pose_NOSE_y",           "pose_NOSE_z"),
    }

    DIST_PAIRS = {
        "dist_wrist_lr":                      ("L_WRIST",   "R_WRIST"),
        "dist_left_wrist_to_left_shoulder":   ("L_WRIST",   "L_SHOULDER"),
        "dist_right_wrist_to_right_shoulder": ("R_WRIST",   "R_SHOULDER"),
        "dist_left_wrist_to_nose":            ("L_WRIST",   "NOSE"),
        "dist_right_wrist_to_nose":           ("R_WRIST",   "NOSE"),
        "dist_left_wrist_to_right_shoulder":  ("L_WRIST",   "R_SHOULDER"),
        "dist_right_wrist_to_left_shoulder":  ("R_WRIST",   "L_SHOULDER"),
        "dist_elbows_lr":                     ("L_ELBOW",   "R_ELBOW"),
    }

    def dist3d(p1_key, p2_key):
        x1, y1, z1 = COLS[p1_key]
        x2, y2, z2 = COLS[p2_key]
        if not all(c in df.columns for c in (x1, y1, z1, x2, y2, z2)):
            return pd.Series(np.nan, index=df.index)
        a = df[[x1, y1, z1]].to_numpy(dtype=float)
        b = df[[x2, y2, z2]].to_numpy(dtype=float)
        return np.linalg.norm(a - b, axis=1)

    def dist3d_pointwise(point_key):
        x, y, z = COLS[point_key]
        if not all(c in df.columns for c in (x, y, z)):
            return pd.Series(np.nan, index=df.index)
        p    = df[[x, y, z]].to_numpy(dtype=float)
        diff = np.linalg.norm(np.diff(p, axis=0), axis=1)
        return pd.Series(np.concatenate([[0.0], diff]), index=df.index)

    for out_col, (p1, p2) in DIST_PAIRS.items():
        df[out_col] = dist3d(p1, p2)

    for point_key in ["L_WRIST", "R_WRIST", "L_SHOULDER", "R_SHOULDER", "NOSE"]:
        df[f"{point_key}_accum_dist"] = dist3d_pointwise(point_key).cumsum()

    df.to_csv(csv_path, index=False)
    print(f"Added distance: {os.path.basename(csv_path)}")
    return df

In [ ]:
def select_peak_features(df: pd.DataFrame) -> list[str]:
    cols: list[str] = []
    cols.extend([c for c in BLENDSHAPES if c in df.columns])
    for base in POSE_KEYS + HAND_WRISTS:
        for axis in ("_x", "_y", "_z"):
            c = f"{base}{axis}"
            if c in df.columns:
                cols.append(c)
    cols.extend([c for c in ANGLES    if c in df.columns])
    cols.extend([c for c in DISTANCES if c in df.columns])

    def is_excluded(name: str) -> bool:
        return (
            name.startswith("transform_")
            or name.endswith("_accum_dist")
            or name.endswith("_velocity")
            or name.endswith("_acceleration")
        )

    seen: set[str] = set()
    out:  list[str] = []
    for c in cols:
        if c not in seen and not is_excluded(c):
            out.append(c)
            seen.add(c)
    return out


def add_peak_columns_inplace(csv_path: str | Path, *,
                              prominence: float | None = None,
                              overwrite: bool = True) -> None:
    csv_path = Path(csv_path)
    df       = pd.read_csv(csv_path)

    for feat in select_peak_features(df):
        x = pd.to_numeric(df[feat], errors="coerce").to_numpy(dtype=float)
        if np.isnan(x).any():
            continue
        peaks, _ = find_peaks(x, prominence=prominence)
        ind       = np.zeros(len(df), dtype=int)
        ind[peaks] = 1
        df[f"{feat}__peak"] = ind

    out_path = csv_path if overwrite else csv_path.with_name(csv_path.stem + "_with_peaks.csv")
    df.to_csv(out_path, index=False)

In [ ]:
def flatten_from_interpolated_csv(csv_path: str, video_name: str,
                                   output_folder: str, meta: dict | None) -> None:
    df           = pd.read_csv(csv_path)
    duration_sec = meta.get("duration_sec", np.nan) if meta else np.nan

    rotation_cols = [c for c in ANGLES    if c in df.columns]
    distance_cols = [c for c in DISTANCES if c in df.columns]
    motion_cols   = [c for c in df.columns if c.endswith("_accum_dist")]

    def summarize_meanstd(cols):
        out = {}
        for col in cols:
            s = pd.to_numeric(df[col], errors="coerce")
            out[f"{col}_mean"] = s.mean()
            out[f"{col}_std"]  = s.std()
        return out

    def summarize_avg(cols):
        return {
            f"{col}_avg": pd.to_numeric(df[col], errors="coerce").mean()
            for col in cols
        }

    # Compute velocity/acceleration then summarise landmarks + blendshapes
    df_with_deriv = compute_derivatives(df, POSE_HAND_LANDMARKS)
    stats_df      = compute_stats(df_with_deriv, POSE_HAND_LANDMARKS, BLENDSHAPES)

    ext = {}
    ext.update(summarize_meanstd(rotation_cols))
    ext.update(summarize_avg(distance_cols))
    ext.update(summarize_avg(motion_cols))

    # Peak rates
    for pc in [c for c in df.columns if c.endswith("__peak")]:
        count = float(pd.to_numeric(df[pc], errors="coerce").fillna(0).sum())
        ext[pc.replace("__peak", "_peaks_per_s")] = (
            count / duration_sec if duration_sec and duration_sec > 0 else np.nan
        )

    combined = pd.concat([stats_df, pd.DataFrame([ext])], axis=1)
    stats_csv = os.path.join(output_folder, f"{video_name}_features.csv")
    combined.to_csv(stats_csv, index=False)
    print(f"Flattened features for ML saved to: {stats_csv}")

In [ ]:
def save_processed_data(df: pd.DataFrame, root_folder: str,
                         video_path: str, meta: dict | None = None) -> None:
    output_folder = os.path.join(
        root_folder, "Output_",
        os.path.dirname(os.path.relpath(video_path, root_folder)),
    )
    os.makedirs(output_folder, exist_ok=True)

    video_name   = os.path.splitext(os.path.basename(video_path))[0]
    output_csv   = os.path.join(output_folder, f"{video_name}_mediapipe_data.csv")
    df.to_csv(output_csv, index=False)
    print(f"Saved raw features: {output_csv}")

    df_interp        = interpolate_missing_values(output_csv)
    interpolated_csv = os.path.join(output_folder, f"{video_name}_mediapipe_data_interpolated.csv")
    df_interp.to_csv(interpolated_csv, index=False)
    print(f"Saved interpolated data: {interpolated_csv}")

    if os.path.exists(interpolated_csv):
        print("Adding rotation & movement features...")
        compute_transform_features(interpolated_csv)

        print("Adding distance and joint motion features...")
        augment_distance(interpolated_csv)

        print("Detecting peaks...")
        add_peak_columns_inplace(interpolated_csv)

        print("Flattening features for ML...")
        flatten_from_interpolated_csv(interpolated_csv, video_name, output_folder, meta)

In [ ]:
root_folder = "/root/"  # change folder

video_files = find_video(root_folder)

for video_file in tqdm(video_files, desc="Processing Videos"):
    df, meta = process_video(video_file)
    save_processed_data(df, root_folder, video_file, meta)